## ⚙️ Cellule 1 — Configuration

In [9]:
from google.colab import drive
import os, json

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive déjà monté')

# ─── CHEMINS (identiques au notebook YOLO, sauf SAVE_DIR / dossier de données local) ──────
FRACATLAS_ZIP    = '/content/drive/MyDrive/FracAtlas/archive.zip'
FRACATLAS_ROOT   = '/content/FracAtlas_extracted'
IMAGES_SUBDIRS   = ['Fractured', 'Non_fractured']
LABELS_DIR       = os.path.join(FRACATLAS_ROOT, 'Annotations', 'YOLO')
SPLITS_PATH      = '/content/drive/MyDrive/resultats_finaux_fracture/splits_fracatlas.json'
STATS_PATH       = '/content/drive/MyDrive/resultats_finaux_fracture/fracatlas_clahe_stats.json'
SAVE_DIR         = '/content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN'
# Nom de variable "YOLO_DATA" conservé volontairement : les cellules 1bis/2/3/3bis ci-dessous
# sont recopiées telles quelles depuis le notebook YOLO et y font référence directement.
# Dossier local différent de celui de YOLO pour ne pas mélanger les deux jeux prétraités.
YOLO_DATA        = '/content/frcnn_fracatlas_dataset'
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── CONFIG — mêmes données / mêmes seeds que YOLOv11n (comparabilité de l'étude) ─────────
IMG_SIZE      = 640
EPOCHS        = 150
PATIENCE      = 20          # aligné sur le run YOLO déjà effectué (comparabilité de l'étude) — le §4.4 du protocole thèse prévoit 15, mais 20 est la valeur réellement utilisée pour YOLO ; gardée ici pour ne pas comparer YOLO et les autres modèles avec des critères d'arrêt différents. À documenter comme valeur spécifique à ce volet article/FracAtlas.
WEIGHT_DECAY  = 0.0005      # = 5e-4, §4.4
AUG_DEGREES   = 8.0         # aligné sur ce qui a réellement tourné pour YOLO (comparabilité)
AUG_TRANSLATE = 0.1
AUG_SCALE     = 0.15

OPTIMIZER  = 'SGD'
MOMENTUM   = 0.9
LR0        = 5e-4           # §4.4
GRAD_CLIP_MAX_NORM = 5.0    # §4.4 — ici directement applicable (boucle d'entraînement manuelle,
                             # contrairement à ultralytics qui ne l'expose pas)
BATCH_SIZE = 2               # §4.4 : batch=2 pour Faster R-CNN (contrainte VRAM T4 à 640px)

CLAHE_CLIP    = 2.0
CLAHE_GRID    = (8, 8)
LABEL_TO_NAME = {0: 'Fracture'}   # classe unique + fond (background) = 2 classes pour torchvision

# Mêmes 5 seeds que YOLO — l'étude de stabilité compare les mêmes graines sur toutes les architectures
SEEDS = [42, 123, 2024, 7, 77]

with open(SPLITS_PATH) as f:
    splits = json.load(f)
with open(STATS_PATH) as f:
    clahe_stats = json.load(f)

print(f"Train : {len(splits['train'])} images | Val : {len(splits['val'])} images | Test : {len(splits['test'])} images")
print(f"CLAHE stats (calculées sur {clahe_stats['n_images']} images train) : mean={clahe_stats['mean'][0]:.4f}, std={clahe_stats['std'][0]:.4f}")
print(f"\nIMG_SIZE={IMG_SIZE} | EPOCHS={EPOCHS} | PATIENCE={PATIENCE} | BATCH_SIZE={BATCH_SIZE} | SEEDS={SEEDS}")


Drive déjà monté
Train : 2857 images | Val : 613 images | Test : 613 images
CLAHE stats (calculées sur 2857 images train) : mean=0.2116, std=0.0552

IMG_SIZE=640 | EPOCHS=150 | PATIENCE=20 | BATCH_SIZE=2 | SEEDS=[42, 123, 2024, 7, 77]


## 📦 Cellule 1bis — Extraction de l'archive FracAtlas

*(identique au notebook YOLO)*

In [10]:
import zipfile

if os.path.isdir(FRACATLAS_ROOT) and len(os.listdir(FRACATLAS_ROOT)) > 0:
    print(f'✅ Déjà extrait dans cette session → {FRACATLAS_ROOT} ({len(os.listdir(FRACATLAS_ROOT))} éléments)')
else:
    assert os.path.exists(FRACATLAS_ZIP), f'Archive introuvable : {FRACATLAS_ZIP}'
    os.makedirs(FRACATLAS_ROOT, exist_ok=True)
    print(f'📦 Extraction de {FRACATLAS_ZIP} vers {FRACATLAS_ROOT}...')
    with zipfile.ZipFile(FRACATLAS_ZIP, 'r') as zf:
        zf.extractall(FRACATLAS_ROOT)
    print('✅ Extraction terminée.')

# Aperçu de l'arborescence extraite, pour confirmer/ajuster IMAGES_SUBDIRS et LABELS_DIR si besoin
print('\nContenu de premier niveau :', os.listdir(FRACATLAS_ROOT))
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    depth = root[len(FRACATLAS_ROOT):].count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + '/', f'({len(files)} fichiers)' if files else '')


✅ Déjà extrait dans cette session → /content/FracAtlas_extracted (1 éléments)

Contenu de premier niveau : ['FracAtlas']
FracAtlas_extracted/ 
  FracAtlas/ (1 fichiers)
    Annotations/ 
    Utilities/ (2 fichiers)
    images/ 


## 🔎 Cellule 2 — Vérification de la structure sur Drive

*(identique au notebook YOLO)*

In [11]:
# Vérifie la présence des dossiers images et labels, et échantillonne quelques fichiers
for sub in IMAGES_SUBDIRS:
    d = os.path.join(FRACATLAS_ROOT, 'images', sub)
    status = '✅' if os.path.isdir(d) else '⚠️  introuvable'
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'{status}  images/{sub}  ({n} fichiers)  →  {d}')

status_lbl = '✅' if os.path.isdir(LABELS_DIR) else '⚠️  introuvable'
n_lbl = len(os.listdir(LABELS_DIR)) if os.path.isdir(LABELS_DIR) else 0
print(f'{status_lbl}  Annotations/YOLO  ({n_lbl} fichiers)  →  {LABELS_DIR}')

# Vérifie qu'un id du split train se retrouve bien quelque part (image + label)
sample_id = splits['train'][0]
stem = os.path.splitext(sample_id)[0]
found_img = None
for sub in IMAGES_SUBDIRS:
    p = os.path.join(FRACATLAS_ROOT, 'images', sub, sample_id)
    if os.path.exists(p):
        found_img = p
        break
found_lbl = os.path.join(LABELS_DIR, stem + '.txt')

print(f"\nTest sur '{sample_id}' :")
print(f'  image : {"✅ trouvée → " + found_img if found_img else "⚠️ NON trouvée dans " + str(IMAGES_SUBDIRS)}')
print(f'  label : {"✅ trouvé" if os.path.exists(found_lbl) else "⚠️ NON trouvé"} → {found_lbl}')
if os.path.exists(found_lbl):
    with open(found_lbl) as f:
        print(f'    contenu : {f.read().strip()}')
    print("    (format attendu : 'class_id x_center y_center width height', normalisé 0-1)")

print("\n⚠️ Si l'un des points ci-dessus échoue, corrige IMAGES_SUBDIRS / LABELS_DIR en Cellule 1 avant de continuer.")


⚠️  introuvable  images/Fractured  (0 fichiers)  →  /content/FracAtlas_extracted/images/Fractured
⚠️  introuvable  images/Non_fractured  (0 fichiers)  →  /content/FracAtlas_extracted/images/Non_fractured
⚠️  introuvable  Annotations/YOLO  (0 fichiers)  →  /content/FracAtlas_extracted/Annotations/YOLO

Test sur 'IMG0003173.jpg' :
  image : ⚠️ NON trouvée dans ['Fractured', 'Non_fractured']
  label : ⚠️ NON trouvé → /content/FracAtlas_extracted/Annotations/YOLO/IMG0003173.txt

⚠️ Si l'un des points ci-dessus échoue, corrige IMAGES_SUBDIRS / LABELS_DIR en Cellule 1 avant de continuer.


## 🔧 Cellule 3 — Prétraitement CLAHE + Letterbox + réécriture des labels YOLO

*(identique au notebook YOLO — mêmes images/labels produits par construction, CLAHE et letterbox étant déterministes)*

In [12]:
import cv2
import numpy as np
from tqdm.auto import tqdm

# ── Index robuste des images : balayage complet de FRACATLAS_ROOT ──
# Plutôt que de supposer une arborescence fixe (images/Fractured, images/Non_fractured...),
# on indexe UNE FOIS tous les fichiers image trouvés sous FRACATLAS_ROOT, par nom de fichier.
# Ça évite de dépendre d'une structure de dossiers qu'on ne connaît pas avec certitude.
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

print(f'🔎 Indexation des images sous {FRACATLAS_ROOT}...')
_image_index = {}
_n_walked = 0
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    for fname in files:
        if os.path.splitext(fname)[1].lower() in IMG_EXTENSIONS:
            _image_index[fname] = os.path.join(root, fname)
            _n_walked += 1
print(f'✅ {_n_walked} fichiers image indexés sous {FRACATLAS_ROOT}')

# Vérification immédiate sur un échantillon du split train (fail-fast si toujours 0 match)
_sample = splits['train'][:5]
_found = [i for i in _sample if i in _image_index]
print(f'   Échantillon train : {len(_found)}/{len(_sample)} trouvés → {_found}')
if len(_found) == 0:
    raise RuntimeError(
        f"Aucune image du split train n'a été retrouvée sous {FRACATLAS_ROOT}. "
        f"Vérifie FRACATLAS_ROOT en Cellule 1 (le dossier racine est-il le bon ?), "
        f"ou que le nom exact des fichiers (ex: '{_sample[0]}') correspond bien à ce qui est sur le Drive."
    )

def find_image_path(image_id):
    return _image_index.get(image_id, None)

# ── CORRECTIF : index robuste des LABELS, même logique que pour les images ──
# LABELS_DIR (Cellule 1) est un chemin statique qui suppose FRACATLAS_ROOT/Annotations/YOLO.
# Si le zip extrait une arborescence imbriquée (ex: FRACATLAS_ROOT/FracAtlas/Annotations/YOLO),
# ce chemin statique ne pointe vers rien, et l'ancien code traitait ça silencieusement comme
# "pas de label" pour TOUTES les images (bug identifié : 100% des images en background).
# On corrige en indexant tous les .txt trouvés sous FRACATLAS_ROOT par nom de fichier,
# restreint aux chemins contenant 'Annotations' pour éviter de capter un .txt parasite ailleurs.
print(f'🔎 Indexation des labels YOLO sous {FRACATLAS_ROOT}...')
_label_index = {}
for root, dirs, files in os.walk(FRACATLAS_ROOT):
    if 'Annotations' not in root:
        continue
    for fname in files:
        if fname.endswith('.txt'):
            _label_index[fname] = os.path.join(root, fname)
print(f'✅ {len(_label_index)} fichiers .txt indexés sous Annotations/ (attendu : ~4083)')

def find_label_path(stem):
    return _label_index.get(stem + '.txt', None)

# Vérification fail-fast : on cherche spécifiquement des images FRACTURÉES du train
# (pas juste les 5 premières, qui sont probablement non-fracturées et auraient un label
# vide de toute façon — ça ne prouverait rien sur le bug qu'on corrige ici)
_fractured_sample = [
    i for i in splits['train']
    if find_image_path(i) and 'Fractured' in find_image_path(i) and 'Non_fractured' not in find_image_path(i)
][:5]
_labels_found = 0
for _fid in _fractured_sample:
    _stem = os.path.splitext(_fid)[0]
    _lp = find_label_path(_stem)
    if _lp and os.path.getsize(_lp) > 0:
        _labels_found += 1
print(f'   Échantillon train FRACTURÉ : {_labels_found}/{len(_fractured_sample)} ont un label non-vide '
      f'{_fractured_sample}')
if len(_fractured_sample) > 0 and _labels_found == 0:
    raise RuntimeError(
        "Aucun label non-vide trouvé pour des images fracturées du train. "
        "Le chemin Annotations/YOLO est peut-être introuvable dans l'arborescence extraite, "
        "ou le nommage des fichiers .txt ne correspond pas aux noms d'images. "
        "Vérifie la sortie de l'indexation ci-dessus (nombre de .txt trouvés)."
    )

def apply_clahe(img_gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    return clahe.apply(img_gray)

def letterbox_image(img, size=IMG_SIZE):
    """Redimensionne en conservant le ratio + padding centré. Retourne l'image + les paramètres
    de transformation (scale, pad_x, pad_y) nécessaires pour ajuster les boîtes."""
    h, w = img.shape[:2]
    scale = min(size / h, size / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_y, pad_x = (size - new_h) // 2, (size - new_w) // 2
    canvas = np.full((size, size), 114, dtype=np.uint8)  # gris neutre, convention YOLO
    canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = resized
    return canvas, scale, pad_x, pad_y

def letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, size=IMG_SIZE):
    """Relit un fichier de labels YOLO (coords normalisées / image d'origine) et renvoie les
    lignes YOLO recalculées pour l'image letterboxée."""
    if label_path is None or not os.path.exists(label_path):
        return []  # image sans fracture annotée (ne devrait pas arriver si Fractured/ contient une box)
    new_lines = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, bw, bh = parts
            xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
            # normalisé -> pixels absolus (image d'origine)
            xc_px, yc_px = xc * orig_w, yc * orig_h
            bw_px, bh_px = bw * orig_w, bh * orig_h
            # applique le même scale + padding que l'image
            xc_new = xc_px * scale + pad_x
            yc_new = yc_px * scale + pad_y
            bw_new = bw_px * scale
            bh_new = bh_px * scale
            # renormalise par IMG_SIZE
            new_lines.append(f'{cls} {xc_new/size:.6f} {yc_new/size:.6f} {bw_new/size:.6f} {bh_new/size:.6f}')
    return new_lines

def preprocess_split(image_ids, split_name):
    img_out_dir = os.path.join(YOLO_DATA, 'images', split_name)
    lbl_out_dir = os.path.join(YOLO_DATA, 'labels', split_name)
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    n_missing_img, n_no_label, n_with_label = 0, 0, 0
    for image_id in tqdm(image_ids, desc=f'Prétraitement {split_name}'):
        stem = os.path.splitext(image_id)[0]
        src_path = find_image_path(image_id)
        if src_path is None:
            n_missing_img += 1
            continue

        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = img.shape[:2]
        img_clahe = apply_clahe(img)
        img_lb, scale, pad_x, pad_y = letterbox_image(img_clahe, IMG_SIZE)
        img_rgb = cv2.cvtColor(img_lb, cv2.COLOR_GRAY2BGR)  # ultralytics attend du 3-canaux
        cv2.imwrite(os.path.join(img_out_dir, stem + '.jpg'), img_rgb)

        label_path = find_label_path(stem)  # CORRECTIF : index robuste au lieu du chemin statique
        new_lines = letterbox_yolo_boxes(label_path, orig_w, orig_h, scale, pad_x, pad_y, IMG_SIZE)
        if label_path is None:
            n_no_label += 1
        elif new_lines:
            n_with_label += 1
        with open(os.path.join(lbl_out_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(new_lines))  # vide = image négative (Non_fractured), convention YOLO

    print(f'  ✅ {split_name} : {len(image_ids) - n_missing_img}/{len(image_ids)} images traitées '
          f'({n_missing_img} introuvables, {n_no_label} sans fichier .txt trouvé, '
          f'{n_with_label} avec au moins une bbox)')

for split_name, ids in [('train', splits['train']), ('val', splits['val']), ('test', splits['test'])]:
    preprocess_split(ids, split_name)


🔎 Indexation des images sous /content/FracAtlas_extracted...
✅ 4083 fichiers image indexés sous /content/FracAtlas_extracted
   Échantillon train : 5/5 trouvés → ['IMG0003173.jpg', 'IMG0000063.jpg', 'IMG0002055.jpg', 'IMG0002154.jpg', 'IMG0001769.jpg']
🔎 Indexation des labels YOLO sous /content/FracAtlas_extracted...
✅ 4084 fichiers .txt indexés sous Annotations/ (attendu : ~4083)
   Échantillon train FRACTURÉ : 5/5 ont un label non-vide ['IMG0002607.jpg', 'IMG0002483.jpg', 'IMG0002577.jpg', 'IMG0002127.jpg', 'IMG0001494.jpg']


Prétraitement train:   0%|          | 0/2857 [00:00<?, ?it/s]

  ✅ train : 2857/2857 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 513 avec au moins une bbox)


Prétraitement val:   0%|          | 0/613 [00:00<?, ?it/s]

  ✅ val : 613/613 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 96 avec au moins une bbox)


Prétraitement test:   0%|          | 0/613 [00:00<?, ?it/s]

  ✅ test : 613/613 images traitées (0 introuvables, 0 sans fichier .txt trouvé, 108 avec au moins une bbox)


## ⚖️ Cellule 3bis — Suréchantillonnage des images fracturées (train uniquement)

*(identique au notebook YOLO — même TARGET_POSITIVE_RATIO=0.37, mêmes fichiers dupliqués)*

In [13]:
import shutil

TARGET_POSITIVE_RATIO = 0.37  # ~35-40%, choix validé (cf. cellule markdown ci-dessus)

train_img_dir = os.path.join(YOLO_DATA, 'images', 'train')
train_lbl_dir = os.path.join(YOLO_DATA, 'labels', 'train')

# Identifie les images positives (label non vide) vs négatives dans le train prétraité
all_stems = [os.path.splitext(f)[0] for f in os.listdir(train_img_dir)]
pos_stems, neg_stems = [], []
for stem in all_stems:
    lbl_path = os.path.join(train_lbl_dir, stem + '.txt')
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        pos_stems.append(stem)
    else:
        neg_stems.append(stem)

n_pos, n_neg = len(pos_stems), len(neg_stems)
current_ratio = n_pos / (n_pos + n_neg)
print(f'Avant suréchantillonnage : {n_pos} positifs / {n_pos + n_neg} total ({current_ratio:.1%})')

# k = nombre total de copies par image positive (originale incluse) pour atteindre TARGET_POSITIVE_RATIO
# dérivé de : n_pos*k / (n_pos*k + n_neg) = TARGET_POSITIVE_RATIO
k = (TARGET_POSITIVE_RATIO * n_neg) / (n_pos * (1 - TARGET_POSITIVE_RATIO))
k_extra = max(0, round(k) - 1)  # nombre de duplicatas à ajouter par image positive
print(f'Facteur de duplication calculé : k={k:.2f} → {k_extra} copie(s) supplémentaire(s) par image positive')

n_created = 0
for stem in pos_stems:
    src_img = os.path.join(train_img_dir, stem + '.jpg')
    src_lbl = os.path.join(train_lbl_dir, stem + '.txt')
    for dup_i in range(1, k_extra + 1):
        dst_img = os.path.join(train_img_dir, f'{stem}_dup{dup_i}.jpg')
        dst_lbl = os.path.join(train_lbl_dir, f'{stem}_dup{dup_i}.txt')
        if not os.path.exists(dst_img):
            shutil.copyfile(src_img, dst_img)
            shutil.copyfile(src_lbl, dst_lbl)
            n_created += 1

n_pos_after = n_pos * (1 + k_extra)
n_total_after = n_pos_after + n_neg
print(f'✅ {n_created} paires image/label créées par duplication (train uniquement)')
print(f'Après suréchantillonnage : {n_pos_after} positifs / {n_total_after} total ({n_pos_after / n_total_after:.1%})')
print('⚠️ val/test non touchés — distribution naturelle conservée pour une évaluation honnête.')

Avant suréchantillonnage : 1539 positifs / 3883 total (39.6%)
Facteur de duplication calculé : k=0.89 → 0 copie(s) supplémentaire(s) par image positive
✅ 0 paires image/label créées par duplication (train uniquement)
Après suréchantillonnage : 1539 positifs / 3883 total (39.6%)
⚠️ val/test non touchés — distribution naturelle conservée pour une évaluation honnête.


## 📦 Cellule 4 — Dataset PyTorch (lecture du format YOLO-txt déjà prétraité) + augmentation

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import v2 as T
import cv2, numpy as np, random, os

# ── Dataset : lit les images/labels déjà prétraités (CLAHE+letterbox+suréchantillonnage,
# Cellules 3/3bis) au format YOLO-txt, et convertit les boîtes en [x1,y1,x2,y2] pixels
# attendu par torchvision.models.detection.* ──
class FracAtlasDetectionDataset(Dataset):
    def __init__(self, split_name, img_size=IMG_SIZE, train=False):
        self.img_dir = os.path.join(YOLO_DATA, 'images', split_name)
        self.lbl_dir = os.path.join(YOLO_DATA, 'labels', split_name)
        self.img_size = img_size
        self.train = train
        self.stems = sorted(os.path.splitext(f)[0] for f in os.listdir(self.img_dir))

    def __len__(self):
        return len(self.stems)

    def _load_boxes(self, stem):
        lbl_path = os.path.join(self.lbl_dir, stem + '.txt')
        boxes, labels = [], []
        if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, xc, yc, bw, bh = parts
                    xc, yc, bw, bh = float(xc) * self.img_size, float(yc) * self.img_size, \
                                       float(bw) * self.img_size, float(bh) * self.img_size
                    x1, y1 = xc - bw / 2, yc - bh / 2
                    x2, y2 = xc + bw / 2, yc + bh / 2
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)  # 0 = background pour torchvision, donc Fracture = 1
        return boxes, labels

    def _augment(self, img, boxes):
        # Augmentation géométrique portable, en miroir des paramètres utilisés pour YOLO
        # (AUG_DEGREES, AUG_TRANSLATE, AUG_SCALE, HFlip p=0.5) — implémentation manuelle
        # car torchvision n'a pas d'équivalent direct au pipeline ultralytics.
        h, w = img.shape[:2]
        boxes = np.array(boxes, dtype=np.float32) if len(boxes) else np.zeros((0, 4), dtype=np.float32)

        if random.random() < 0.5:  # fliplr=0.5, aligné §4.3 / YOLO
            img = cv2.flip(img, 1)
            if len(boxes):
                boxes[:, [0, 2]] = w - boxes[:, [2, 0]]

        angle = random.uniform(-AUG_DEGREES, AUG_DEGREES)
        scale = 1.0 + random.uniform(-AUG_SCALE, AUG_SCALE)
        tx = random.uniform(-AUG_TRANSLATE, AUG_TRANSLATE) * w
        ty = random.uniform(-AUG_TRANSLATE, AUG_TRANSLATE) * h
        center = (w / 2, h / 2)
        M = cv2.getRotationMatrix2D(center, angle, scale)
        M[0, 2] += tx
        M[1, 2] += ty
        img = cv2.warpAffine(img, M, (w, h), borderValue=(114, 114, 114))

        if len(boxes):
            corners = np.zeros((len(boxes), 4, 2), dtype=np.float32)
            corners[:, 0] = boxes[:, [0, 1]]
            corners[:, 1] = boxes[:, [2, 1]]
            corners[:, 2] = boxes[:, [2, 3]]
            corners[:, 3] = boxes[:, [0, 3]]
            corners = corners.reshape(-1, 2)
            ones = np.ones((corners.shape[0], 1), dtype=np.float32)
            corners_h = np.hstack([corners, ones])
            corners_t = (M @ corners_h.T).T.reshape(-1, 4, 2)
            x1 = corners_t[:, :, 0].min(axis=1).clip(0, w)
            y1 = corners_t[:, :, 1].min(axis=1).clip(0, h)
            x2 = corners_t[:, :, 0].max(axis=1).clip(0, w)
            y2 = corners_t[:, :, 1].max(axis=1).clip(0, h)
            boxes = np.stack([x1, y1, x2, y2], axis=1)
            # Retire les boîtes devenues dégénérées (hors image après transformation)
            keep = (boxes[:, 2] - boxes[:, 0] > 2) & (boxes[:, 3] - boxes[:, 1] > 2)
            boxes = boxes[keep]
        return img, boxes

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img = cv2.imread(os.path.join(self.img_dir, stem + '.jpg'))  # déjà CLAHE+letterbox (Cellule 3)
        boxes, labels = self._load_boxes(stem)

        if self.train:
            img, boxes = self._augment(img, np.array(boxes) if boxes else [])
            labels = labels[:len(boxes)]  # aligné après filtrage des boîtes dégénérées

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        # Normalisation — mêmes stats CLAHE que le reste du protocole (fichier stats_clahe)
        mean = np.array(clahe_stats['mean'], dtype=np.float32)
        std = np.array(clahe_stats['std'], dtype=np.float32)
        img = (img - mean) / std
        img_t = torch.from_numpy(img.transpose(2, 0, 1)).float()

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.as_tensor(labels, dtype=torch.int64).reshape(-1)
        target = {
            'boxes': boxes_t,
            'labels': labels_t,
            'image_id': torch.tensor([idx]),
            'stem': stem,
        }
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))


## 🚀 Cellule 5 — Entraînement multi-seed Faster R-CNN (ResNet50 FPN v2)

Boucle manuelle (torchvision ne fournit pas de `.train()` haut niveau comme ultralytics) : optimizer SGD, warmup + cosine annealing, gradient clipping, AMP FP16, early stopping sur la loss de validation (patience=15), reprise depuis checkpoint si déjà entraîné — même convention que YOLO.

In [15]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics'], check=True)

import torch, gc, random, math, json as _json
import numpy as np
from torch.optim.lr_scheduler import LambdaLR
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = 2  # background + Fracture

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

def build_model():
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return model.to(device)

def warmup_cosine_lr(step, warmup_steps, total_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def validate_loss(model, val_loader):
    # torchvision ne renvoie une loss qu'en mode train() ; on force ce mode pour le calcul
    # de la loss de validation (sans mise à jour des poids), pratique standard avec cette API.
    model.train()
    total_loss = 0.0
    n = 0
    for imgs, targets in val_loader:
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items() if k != 'stem'} for t in targets]
        with torch.no_grad():
            loss_dict = model(imgs, targets)
        total_loss += sum(loss_dict.values()).item()
        n += 1
    return total_loss / max(1, n)

train_ds = FracAtlasDetectionDataset('train', train=True)
val_ds   = FracAtlasDetectionDataset('val', train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

trained_weights = {}  # seed -> chemin vers best.pth

for seed in SEEDS:
    seed_dir = os.path.join(SAVE_DIR, f'seed_{seed}')
    os.makedirs(seed_dir, exist_ok=True)
    best_ckpt = os.path.join(seed_dir, 'best.pth')
    resume_ckpt = os.path.join(seed_dir, 'resume_state.pth')      # état complet pour reprendre en cours de route
    done_marker = os.path.join(seed_dir, 'training_done.json')     # écrit UNIQUEMENT quand l'entraînement va au bout

    # ── Cas 1 : entraînement déjà complet (marqueur présent) → on charge et on passe au seed suivant ──
    if os.path.exists(best_ckpt) and os.path.exists(done_marker):
        print(f'✅ Seed {seed} : entraînement complet, chargement → {best_ckpt}')
        trained_weights[seed] = best_ckpt
        continue

    set_seed(seed)
    model = build_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))  # AMP FP16, §4.4

    steps_per_epoch = len(train_loader)
    warmup_steps = 500  # §4.4 : warmup 500 iters (ici applicable littéralement, boucle manuelle)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: warmup_cosine_lr(s, warmup_steps, total_steps))

    start_epoch = 0
    best_val_loss = float('inf')
    patience_counter = 0
    global_step = 0

    # ── Cas 2 : run interrompu détecté (resume_state.pth présent mais pas de marqueur "done") → on reprend ──
    if os.path.exists(resume_ckpt):
        print(f'\n🔄 Seed {seed} — entraînement incomplet détecté, reprise depuis {resume_ckpt}...')
        state = torch.load(resume_ckpt, map_location=device)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        scheduler.load_state_dict(state['scheduler'])
        start_epoch = state['epoch'] + 1
        best_val_loss = state['best_val_loss']
        patience_counter = state['patience_counter']
        global_step = state['global_step']
        print(f'   → reprise à l\'epoch {start_epoch}/{EPOCHS} (best_val_loss={best_val_loss:.4f}, patience={patience_counter}/{PATIENCE})')
    else:
        print(f'\n🚀 Seed {seed} — entraînement Faster R-CNN ResNet50 FPN v2 (640px)...')

    training_completed_normally = False

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0
        for imgs, targets in train_loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items() if k != 'stem'} for t in targets]

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                loss_dict = model(imgs, targets)
                loss = sum(loss_dict.values())

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)  # §4.4
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_loss += loss.item()
            global_step += 1

        val_loss = validate_loss(model, val_loader)
        print(f'  Epoch {epoch+1}/{EPOCHS} — train_loss={epoch_loss/steps_per_epoch:.4f} | val_loss={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_ckpt)
        else:
            patience_counter += 1

        # ── Sauvegarde de l'état complet à CHAQUE epoch, pour pouvoir reprendre si la session est coupée ──
        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scaler': scaler.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch,
            'best_val_loss': best_val_loss,
            'patience_counter': patience_counter,
            'global_step': global_step,
        }, resume_ckpt)

        if patience_counter >= PATIENCE:  # 15, §4.4
            print(f'  ⏹️ Early stopping — patience {PATIENCE} atteinte (meilleure val_loss={best_val_loss:.4f})')
            training_completed_normally = True
            break
    else:
        # la boucle for est allée au bout sans break → toutes les epochs ont été faites
        training_completed_normally = True

    if training_completed_normally:
        with open(done_marker, 'w') as f:
            _json.dump({'seed': seed, 'best_val_loss': best_val_loss, 'epochs_done': epoch + 1}, f)
        # état de reprise devenu inutile une fois l'entraînement terminé
        if os.path.exists(resume_ckpt):
            os.remove(resume_ckpt)

    trained_weights[seed] = best_ckpt
    print(f'✅ Seed {seed} — terminé, meilleur checkpoint → {best_ckpt}')

    del model, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n🏁 Entraînement terminé pour les {len(SEEDS)} seeds.")

✅ Seed 42 : entraînement complet, chargement → /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/seed_42/best.pth
✅ Seed 123 : entraînement complet, chargement → /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/seed_123/best.pth
✅ Seed 2024 : entraînement complet, chargement → /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/seed_2024/best.pth
✅ Seed 7 : entraînement complet, chargement → /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/seed_7/best.pth
✅ Seed 77 : entraînement complet, chargement → /content/drive/MyDrive/resultats_finaux_fracture/Detection/FasterRCNN/seed_77/best.pth

🏁 Entraînement terminé pour les 5 seeds.


## 📊 Cellule 6 — Évaluation multi-seed (AUC, sensibilité, spécificité, mAP50)

Même logique que la Cellule 6 de YOLO : mAP@IoU=0.5 (torchmetrics) + score image-level (confiance max des boîtes prédites) pour AUC/sensibilité/spécificité au seuil de Youden + IC95% bootstrap n=1000.

In [16]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np


test_ds = FracAtlasDetectionDataset('test', train=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2)

results_by_seed = {}

for seed in SEEDS:
    metrics_path = os.path.join(SAVE_DIR, f'metriques_seed{seed}.json')

    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            cached = _json.load(f)
        if 'mAP50' in cached.get('Fracture', {}):
            results_by_seed[seed] = cached
            print(f'✅ Seed {seed} : métriques déjà calculées, chargées depuis le disque.')
            continue

    print(f'\n📊 Seed {seed} — évaluation...')
    model = build_model()
    model.load_state_dict(torch.load(trained_weights[seed], map_location=device))
    model.to(device)
    model.eval()

    model.roi_heads.score_thresh = 0.001
    model.roi_heads.detections_per_img = 300

    # mAP50 (IoU=0.5 uniquement, cohérence avec le protocole thoracique — cf. correctif YOLO/RetinaNet)
    map_metric = MeanAveragePrecision(iou_thresholds=[0.5])
    y_true, y_score = [], []

    with torch.no_grad():
        for imgs, targets in test_loader:
            imgs_dev = [img.to(device) for img in imgs]
            preds = model(imgs_dev)

            preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
            targets_cpu = [{'boxes': t['boxes'], 'labels': t['labels']} for t in targets]
            map_metric.update(preds_cpu, targets_cpu)

            has_fracture = int(len(targets[0]['boxes']) > 0)
            score = float(preds[0]['scores'].max()) if len(preds[0]['scores']) > 0 else 0.0
            y_true.append(has_fracture)
            y_score.append(score)

    map50 = float(map_metric.compute()['map'])
    y_true, y_score = np.array(y_true), np.array(y_score)

    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    youden_idx = np.argmax(tpr - fpr)
    best_thr = float(thresholds[youden_idx])
    sensib = float(tpr[youden_idx])
    specif = float(1 - fpr[youden_idx])

    # Bootstrap IC95% de l'AUC — identique à YOLO (n=1000)
    rng = np.random.default_rng(seed)
    boot_aucs = []
    n = len(y_true)
    for _ in range(1000):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])

    result = {
        'Fracture': {
            'AUC': round(float(auc), 4),
            'IC95': [round(float(ci_low), 4), round(float(ci_high), 4)],
            'Sensibilite': round(sensib, 4),
            'Specificite': round(specif, 4),
            'Seuil_Youden': round(best_thr, 4),
            'mAP50': round(map50, 4),
        },
        'global': {
            'model': 'FasterRCNN_ResNet50_FPN_V2',
            'mAP50': round(map50, 4),
            'seed': seed,
            'img_size': IMG_SIZE,
            'preprocessing': f'CLAHE(clip={CLAHE_CLIP})+Letterbox{IMG_SIZE}',
        },
    }
    with open(metrics_path, 'w') as f:
        _json.dump(result, f, indent=2)
    results_by_seed[seed] = result
    print(f'✅ Seed {seed} — AUC={auc:.4f} | mAP50={map50:.4f}')

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



📊 Seed 42 — évaluation...
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:01<00:00, 146MB/s]
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


✅ Seed 42 — AUC=0.8579 | mAP50=0.3698

📊 Seed 123 — évaluation...
✅ Seed 123 — AUC=0.8561 | mAP50=0.3896

📊 Seed 2024 — évaluation...
✅ Seed 2024 — AUC=0.8281 | mAP50=0.4175

📊 Seed 7 — évaluation...


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


✅ Seed 7 — AUC=0.8655 | mAP50=0.4368

📊 Seed 77 — évaluation...


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


✅ Seed 77 — AUC=0.8522 | mAP50=0.4233


## 🏁 Cellule 7 — Agrégation sur les 5 seeds (écart-type ddof = 1)

In [17]:
aucs = [results_by_seed[s]['Fracture']['AUC'] for s in SEEDS]
maps = [results_by_seed[s]['Fracture']['mAP50'] for s in SEEDS]

agg = {
    'Fracture': {
        'AUC_mean': round(float(np.mean(aucs)), 4),
        'AUC_std': round(float(np.std(aucs, ddof=1)), 4),
        'mAP50_mean': round(float(np.mean(maps)), 4),
        'mAP50_std': round(float(np.std(maps, ddof=1)), 4),
        'n_seeds': len(SEEDS),
    },
    'global': {'model': 'FasterRCNN_ResNet50_FPN_V2', 'seeds': SEEDS},
}
with open(os.path.join(SAVE_DIR, 'resultats_agreges_5seeds.json'), 'w') as f:
    json.dump(agg, f, indent=2)

print(f"AUC   = {agg['Fracture']['AUC_mean']:.4f} ± {agg['Fracture']['AUC_std']:.4f}")
print(f"mAP50 = {agg['Fracture']['mAP50_mean']:.4f} ± {agg['Fracture']['mAP50_std']:.4f}")


AUC   = 0.8520 ± 0.0127
mAP50 = 0.4074 ± 0.0243


## 🧪 Cellule 8 — Contrôle : résolution (640px) et normalisation — 3 seeds

À exécuter après les cellules 1 à 5 (les 5 seeds de référence sont chargés depuis le disque).

- **A_640** : `min_size=640, max_size=640`.
- **B_640_sans_double_norm** : A + `image_mean=0, image_std=1` (normalisation CLAHE seule).
- **C_640_ImageNet_standard** : A + normalisation ImageNet standard (annule algébriquement la normalisation CLAHE du dataset).

3 seeds (42, 123, 2024) par variante, comparés à la référence 5 seeds en écarts-types.

In [18]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONTRÔLE — effet de la résolution interne (800 px) et de la double normalisation
# Faster R-CNN ResNet50 FPN v2 — seed 42 par défaut (CTRL_SEEDS pour en ajouter)
#
# À COLLER DANS UNE NOUVELLE CELLULE du notebook Faster R-CNN, APRÈS les cellules 1 à 5
# (la cellule 5 saute les seeds déjà terminés grâce aux training_done.json, elle définit
#  set_seed, warmup_cosine_lr, validate_loss, train_loader, val_loader utilisés ici).
#
# Trois variantes, tout le reste identique à la référence (SGD, LR 5e-4, batch 2, warmup 500 +
# cosine, clipping 5.0, AMP, patience 20, mêmes splits, même prétraitement, même seed) :
#   A_640                   : min_size=640, max_size=640  → plus d'agrandissement interne
#   B_640_sans_double_norm  : A + image_mean=0, image_std=1 → seule la normalisation CLAHE
#                             du dataset s'applique (ce que décrit l'étape 4 du §3)
#   C_640_ImageNet_standard : A + normalisation ImageNet standard de torchvision appliquée à
#                             l'image brute /255 (la normalisation CLAHE du dataset est annulée
#                             algébriquement via image_mean/image_std, sans toucher au dataset)
# ═══════════════════════════════════════════════════════════════════════════════
import os, json, gc, random, math
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import roc_auc_score, roc_curve
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

CTRL_SEEDS = [42, 123, 2024]   # 3 seeds pour un signal exploitable directement
CTRL_DIR = os.path.join(SAVE_DIR, 'controle_resolution')
os.makedirs(CTRL_DIR, exist_ok=True)

VARIANTS = {
    'A_640': dict(min_size=640, max_size=640),
    'B_640_sans_double_norm': dict(min_size=640, max_size=640,
                                   image_mean=[0.0, 0.0, 0.0], image_std=[1.0, 1.0, 1.0]),
}

# Variante C : le dataset renvoie z = (x - m)/s. On veut que le modèle voie (x - mu)/sigma (ImageNet).
# Le transform interne calcule (z - image_mean)/image_std ; avec image_mean = (mu - m)/s et
# image_std = sigma/s on obtient exactement (x - mu)/sigma.
_m = np.array(clahe_stats['mean'], dtype=np.float64).ravel()
_s = np.array(clahe_stats['std'], dtype=np.float64).ravel()
if _m.size == 1: _m = np.repeat(_m, 3)
if _s.size == 1: _s = np.repeat(_s, 3)
_IM_MEAN = np.array([0.485, 0.456, 0.406]); _IM_STD = np.array([0.229, 0.224, 0.225])
VARIANTS['C_640_ImageNet_standard'] = dict(
    min_size=640, max_size=640,
    image_mean=[float(v) for v in (_IM_MEAN - _m) / _s],
    image_std=[float(v) for v in _IM_STD / _s])

# Optionnel : ré-entraîner la configuration d'origine (800 px + double norm) avec CE code,
# pour mesurer la reproductibilité seed 42 → seed 42 (non-déterminisme GPU). Coûte un run de plus.
RUN_REFERENCE_REPRO = False
if RUN_REFERENCE_REPRO:
    VARIANTS['REF_800_repro'] = dict()


def build_model_variant(**model_kwargs):
    # kwargs (min_size, max_size, image_mean, image_std) sont transmis à GeneralizedRCNNTransform
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT, **model_kwargs)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return model.to(device)


def train_variant(name, model_kwargs, seed):
    vdir = os.path.join(CTRL_DIR, f'{name}_seed{seed}')
    os.makedirs(vdir, exist_ok=True)
    best_ckpt = os.path.join(vdir, 'best.pth')
    resume_ckpt = os.path.join(vdir, 'resume_state.pth')
    done_marker = os.path.join(vdir, 'training_done.json')

    if os.path.exists(best_ckpt) and os.path.exists(done_marker):
        print(f'✅ {name} : entraînement déjà complet → {best_ckpt}')
        return best_ckpt

    set_seed(seed)
    model = build_model_variant(**model_kwargs)
    optimizer = torch.optim.SGD(model.parameters(), lr=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = LambdaLR(optimizer, lr_lambda=lambda s: warmup_cosine_lr(s, 500, total_steps))

    start_epoch, best_val_loss, patience_counter = 0, float('inf'), 0
    if os.path.exists(resume_ckpt):
        state = torch.load(resume_ckpt, map_location=device)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        scheduler.load_state_dict(state['scheduler'])
        start_epoch = state['epoch'] + 1
        best_val_loss = state['best_val_loss']
        patience_counter = state['patience_counter']
        print(f'🔄 {name} : reprise à l\'epoch {start_epoch}')
    else:
        print(f'🚀 {name} — seed {seed} — kwargs modèle : {model_kwargs}')

    epoch = start_epoch - 1
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0
        for imgs, targets in train_loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items() if k != 'stem'} for t in targets]
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                loss = sum(model(imgs, targets).values())
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            epoch_loss += loss.item()

        val_loss = validate_loss(model, val_loader)   # même fonction que la référence
        print(f'  [{name}] epoch {epoch+1}/{EPOCHS} — train={epoch_loss/steps_per_epoch:.4f} | val={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss, patience_counter = val_loss, 0
            torch.save(model.state_dict(), best_ckpt)
        else:
            patience_counter += 1

        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scaler': scaler.state_dict(), 'scheduler': scheduler.state_dict(),
                    'epoch': epoch, 'best_val_loss': best_val_loss,
                    'patience_counter': patience_counter}, resume_ckpt)

        if patience_counter >= PATIENCE:
            print(f'  ⏹️ early stopping (patience {PATIENCE}), meilleure val_loss={best_val_loss:.4f}')
            break

    with open(done_marker, 'w') as f:
        json.dump({'seed': seed, 'variant': name, 'best_val_loss': best_val_loss,
                   'epochs_done': epoch + 1, 'model_kwargs': model_kwargs}, f)
    if os.path.exists(resume_ckpt):
        os.remove(resume_ckpt)

    del model, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_ckpt


@torch.no_grad()
def evaluate_variant(name, model_kwargs, ckpt, seed):
    # Même logique d'évaluation que la cellule 6 (mAP50 torchmetrics, AUC image-level =
    # confiance max, sensibilité/spécificité au seuil de Youden), sur le test set gelé.
    model = build_model_variant(**model_kwargs)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    test_ds_ctrl = FracAtlasDetectionDataset('test', train=False)
    test_loader_ctrl = DataLoader(test_ds_ctrl, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2)

    map_metric = MeanAveragePrecision(iou_thresholds=[0.5])
    y_true, y_score = [], []
    for imgs, targets in test_loader_ctrl:
        preds = model([img.to(device) for img in imgs])
        preds_cpu = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{'boxes': t['boxes'], 'labels': t['labels']} for t in targets]
        map_metric.update(preds_cpu, targets_cpu)
        y_true.append(int(len(targets[0]['boxes']) > 0))
        y_score.append(float(preds[0]['scores'].max()) if len(preds[0]['scores']) > 0 else 0.0)

    map50 = float(map_metric.compute()['map'])
    y_true, y_score = np.array(y_true), np.array(y_score)
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, thr = roc_curve(y_true, y_score)
    j = int(np.argmax(tpr - fpr))
    res = {'variant': name, 'seed': seed, 'model_kwargs': model_kwargs,
           'AUC': round(float(auc), 4), 'mAP50': round(map50, 4),
           'Sensibilite': round(float(tpr[j]), 4), 'Specificite': round(float(1 - fpr[j]), 4),
           'Seuil_Youden': round(float(thr[j]), 4)}
    with open(os.path.join(CTRL_DIR, f'metriques_{name}_seed{seed}.json'), 'w') as f:
        json.dump(res, f, indent=2)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res


# ── Exécution ──────────────────────────────────────────────────────────────────
ctrl_results = {}   # (variante, seed) -> métriques
for seed in CTRL_SEEDS:
    for name, kw in VARIANTS.items():
        mpath = os.path.join(CTRL_DIR, f'metriques_{name}_seed{seed}.json')
        if os.path.exists(mpath):
            with open(mpath) as f:
                ctrl_results[(name, seed)] = json.load(f)
            print(f'✅ {name} seed {seed} : métriques déjà calculées.')
            continue
        ckpt = train_variant(name, kw, seed)
        ctrl_results[(name, seed)] = evaluate_variant(name, kw, ckpt, seed)

# ── Comparaison avec la référence (config d'origine, déjà entraînée) ───────────
ref = {}
for s_ in SEEDS:
    p = os.path.join(SAVE_DIR, f'metriques_seed{s_}.json')
    if os.path.exists(p):
        with open(p) as f:
            m = json.load(f)['Fracture']
        ref[s_] = (m['AUC'], m['mAP50'])

aucs = np.array([v[0] for v in ref.values()])
maps = np.array([v[1] for v in ref.values()])
print('\n' + '=' * 78)
print('RÉFÉRENCE (800 px + double normalisation) — 5 seeds')
for s_, (a, m) in ref.items():
    print(f'  seed {s_:>5} : AUC={a:.4f} | mAP50={m:.4f}')
print(f'  moyenne ± écart-type : AUC={aucs.mean():.4f} ± {aucs.std():.4f} | mAP50={maps.mean():.4f} ± {maps.std():.4f}')

print('\nCONTRÔLES')
for (name, seed), r in ctrl_results.items():
    print(f"  {name:<26} seed {seed:<5} AUC={r['AUC']:.4f} | mAP50={r['mAP50']:.4f} | "
          f"Sens={r['Sensibilite']:.3f} | Spec={r['Specificite']:.3f}")

print('\nÉCARTS vs référence de la MÊME seed (en écarts-types inter-seeds de la référence)')
for (name, seed), r in ctrl_results.items():
    if seed in ref:
        a0, m0 = ref[seed]
        print(f"  {name:<26} seed {seed:<5} ΔAUC={r['AUC']-a0:+.4f} ({(r['AUC']-a0)/max(aucs.std(),1e-9):+.2f} σ) | "
              f"ΔmAP50={r['mAP50']-m0:+.4f} ({(r['mAP50']-m0)/max(maps.std(),1e-9):+.2f} σ)")
print('=' * 78)
print('Lecture : |Δ| > 2 σ → effet réel probable. |Δ| < 1 σ → effet NON DÉTECTABLE avec ce nombre de seeds')
print('(ce n\'est pas une preuve d\'absence d\'effet : avec 1 seed, seuls les gros effets ressortent).')


✅ A_640 seed 42 : métriques déjà calculées.
✅ B_640_sans_double_norm seed 42 : métriques déjà calculées.
✅ C_640_ImageNet_standard seed 42 : métriques déjà calculées.
✅ A_640 seed 123 : métriques déjà calculées.
✅ B_640_sans_double_norm seed 123 : métriques déjà calculées.
✅ C_640_ImageNet_standard seed 123 : métriques déjà calculées.
✅ A_640 seed 2024 : métriques déjà calculées.
✅ B_640_sans_double_norm seed 2024 : métriques déjà calculées.
✅ C_640_ImageNet_standard seed 2024 : métriques déjà calculées.

RÉFÉRENCE (800 px + double normalisation) — 5 seeds
  seed    42 : AUC=0.8579 | mAP50=0.3698
  seed   123 : AUC=0.8561 | mAP50=0.3896
  seed  2024 : AUC=0.8281 | mAP50=0.4175
  seed     7 : AUC=0.8655 | mAP50=0.4368
  seed    77 : AUC=0.8522 | mAP50=0.4233
  moyenne ± écart-type : AUC=0.8520 ± 0.0127 | mAP50=0.4074 ± 0.0243

CONTRÔLES
  A_640                      seed 42    AUC=0.8365 | mAP50=0.3432 | Sens=0.713 | Spec=0.881
  B_640_sans_double_norm     seed 42    AUC=0.8290 | mAP50=0

## 🔬 Cellule — Analyse des erreurs (Faster R-CNN, configuration retenue, 5 seeds)

À exécuter après la Cellule 5 (poids dans `seed_X/best.pth`) et la Cellule 4 (Dataset). Classe chaque fracture du test (détectée / score trop bas / mal localisée / non détectée) et chaque faux positif, avec ventilation par région, taille, matériel et fractures multiples. Seuil de confiance de chaque seed choisi sur la validation (F1 maximal). Résultats écrits dans `resultats_finaux_fracture/Erreurs/`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSE DES ERREURS — Faster R-CNN (ResNet50 FPN v2), configuration retenue, 5 seeds
# Réutilise : SAVE_DIR, YOLO_DATA, SPLITS_PATH, IMG_SIZE, SEEDS, FRACATLAS_ROOT, device,
#             build_model, FracAtlasDetectionDataset (déjà définis plus haut)
# Poids : <SAVE_DIR>/seed_X/best.pth
# Résultats : <resultats_finaux_fracture>/Erreurs/  (CSV + synthèse + images d'exemples)
# ══════════════════════════════════════════════════════════════════════════════
import os, glob, json, pickle, gc
import numpy as np, pandas as pd, cv2, torch
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

MODEL_NAME = 'Faster R-CNN'
ERR_DIR = os.path.join(os.path.dirname(SPLITS_PATH), 'Erreurs'); os.makedirs(ERR_DIR, exist_ok=True)
IOU_THR, MIN_CONF, FIXED_CONF = 0.5, 0.001, 0.25      # IoU = celui du mAP@50 ; seuil fixe si pas de validation
weights = {s: os.path.join(SAVE_DIR, f'seed_{s}', 'best.pth') for s in SEEDS}
for s, p in weights.items():
    assert os.path.exists(p), f'Poids introuvables (seed {s}) : {p}'

# ── Jeux de test et de validation déjà prétraités (CLAHE + letterbox), via le Dataset du notebook ──
def load_split(split):
    ds = FracAtlasDetectionDataset(split, train=False)
    paths = {s: os.path.join(YOLO_DATA, 'images', split, s + '.jpg') for s in ds.stems}
    gts = {s: ds[i][1]['boxes'].numpy().reshape(-1, 4) for i, s in enumerate(ds.stems)}   # xyxy en pixels (640)
    return ds, paths, gts

TEST_DS, TEST_P, TEST_GT = load_split('test'); VAL_DS, VAL_P, VAL_GT = load_split('val')
n_gt = sum(len(g) for g in TEST_GT.values())
print(f"TEST : {len(TEST_P)} images dont {sum(len(g) > 0 for g in TEST_GT.values())} fracturées ({n_gt} fractures annotées)  [attendu : 613 dont 108]")

# ── Appariement (IoU >= 0,5, glouton par score décroissant) ──
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def match(gt, boxes, scores, thr):
    k = scores >= thr; pb, ps = boxes[k], scores[k]
    o = np.argsort(-ps); pb, ps = pb[o], ps[o]
    tp = np.zeros(len(pb), bool); ok = np.zeros(len(gt), bool)
    if len(gt) and len(pb):
        iou = iou_matrix(pb, gt)
        for i in range(len(pb)):
            free = np.where(~ok)[0]
            if len(free) == 0: break
            j = free[np.argmax(iou[i, free])]
            if iou[i, j] >= IOU_THR: ok[j] = True; tp[i] = True
    return pb, ps, tp, ok

def best_threshold(preds, gts):       # seuil de confiance = F1 maximal sur la validation
    best, bt = -1, FIXED_CONF
    for t in np.arange(0.05, 0.96, 0.05):
        TP = FP = FN = 0
        for s, (b, sc) in preds.items():
            _, _, tp, ok = match(gts[s], b, sc, t); TP += tp.sum(); FP += (~tp).sum(); FN += (~ok).sum()
        f1 = 2 * TP / (2 * TP + FP + FN + 1e-9)
        if f1 > best: best, bt = f1, float(t)
    return bt

# ── Prédictions (sauvegardées sur Drive : reprise possible après une coupure) ──
cache_path = os.path.join(ERR_DIR, 'preds_fasterrcnn.pkl')
cache = pickle.load(open(cache_path, 'rb')) if os.path.exists(cache_path) else {'test': {}, 'thr': {}}
_m = [sc.min() for d in cache['test'].values() for (_, sc) in d.values() if len(sc)]
if _m and min(_m) >= 0.05:
    raise AssertionError(f'Cache {cache_path} : score minimal = {min(_m):.3f} >= 0,05 → prédictions faites avec les réglages par défaut '
                         f'de torchvision. Supprime ce fichier et relance (section 2.5 : score >= 0,001, 300 boîtes).')
def predict_all(model, ds, desc):
    out = {}
    with torch.no_grad():
        for i, s in enumerate(tqdm(ds.stems, desc=desc, leave=False)):
            o = model([ds[i][0].to(device)])[0]
            k = o['labels'] == 1
            out[s] = (o['boxes'][k].cpu().numpy(), o['scores'][k].cpu().numpy())
    return out
for seed in SEEDS:
    if seed in cache['thr']: print(f'✅ seed {seed} : prédictions déjà faites'); continue
    print(f'🔎 seed {seed} — prédictions test + validation...')
    model = build_model(); model.load_state_dict(torch.load(weights[seed], map_location=device)); model.to(device).eval()
    model.roi_heads.score_thresh = MIN_CONF; model.roi_heads.detections_per_img = 300      # comme dans l'évaluation (Cellule 6)
    print(f'   sorties : score_thresh={model.roi_heads.score_thresh}, detections_per_img={model.roi_heads.detections_per_img}, nms_thresh={model.roi_heads.nms_thresh}')
    cache['test'][seed] = predict_all(model, TEST_DS, 'test')
    cache['thr'][seed] = best_threshold(predict_all(model, VAL_DS, 'val'), VAL_GT)
    pickle.dump(cache, open(cache_path, 'wb'))
    print(f"   seuil de confiance retenu (F1 max sur la validation) : {cache['thr'][seed]:.2f}")
    del model; gc.collect(); torch.cuda.empty_cache()

# ── Contrôle : l'AUC recalculée ici doit retrouver celle de la cellule d'évaluation ──
for seed in SEEDS:
    mp = os.path.join(SAVE_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(mp):
        y = [int(len(TEST_GT[s]) > 0) for s in TEST_P]
        sc = [float(cache['test'][seed][s][1].max()) if len(cache['test'][seed][s][1]) else 0.0 for s in TEST_P]
        a_ici, a_ref = roc_auc_score(y, sc), json.load(open(mp))['AUC']
        print(f'   contrôle seed {seed} : AUC ici = {a_ici:.4f} | AUC évaluation = {a_ref:.4f}', '✅' if abs(a_ici - a_ref) < 2e-3 else '⚠️ ÉCART')

# ── Métadonnées FracAtlas : région, matériel, fractures multiples ──
gt_df = pd.DataFrame([dict(stem=s, gt_idx=j, rel_area=(b[2] - b[0]) * (b[3] - b[1]) / IMG_SIZE ** 2)
                      for s, g in TEST_GT.items() for j, b in enumerate(g)])
gt_df['taille'] = pd.qcut(gt_df.rel_area.rank(method='first'), 3, labels=['petite', 'moyenne', 'grande']).astype(str)
for c in ('région', 'matériel', 'fractures multiples'): gt_df[c] = 'inconnu'
mc = glob.glob(f'{FRACATLAS_ROOT}/**/dataset.csv', recursive=True)
if mc:
    meta = pd.read_csv(mc[0]); meta['stem'] = meta['image_id'].astype(str).map(lambda x: os.path.splitext(x)[0])
    def _reg(r):
        if r.get('mixed', 0) == 1: return 'mixte'
        for c, lab in (('hand', 'main'), ('shoulder', 'épaule'), ('leg', 'jambe'), ('hip', 'hanche')):
            if r.get(c, 0) == 1: return lab
        return 'inconnu'
    meta['région'] = meta.apply(_reg, axis=1)
    meta['matériel'] = np.where(meta.get('hardware', 0) == 1, 'avec matériel', 'sans matériel')
    meta['fractures multiples'] = np.where(meta.get('fracture_count', 1) > 1, 'plusieurs', 'une seule')
    gt_df = gt_df.drop(columns=['région', 'matériel', 'fractures multiples']).merge(
        meta[['stem', 'région', 'matériel', 'fractures multiples']], on='stem', how='left').fillna('inconnu')
else:
    print('⚠️ dataset.csv introuvable : pas de ventilation par région / matériel')

# ── Classement des erreurs ──
gt_rows, fp_rows, runs = [], [], []
for seed in SEEDS:
    thr, TP, FPn = cache['thr'][seed], 0, 0
    for s, (b, sc) in cache['test'][seed].items():
        g = TEST_GT[s]; pb, ps, tp, ok = match(g, b, sc, thr); TP += int(tp.sum()); FPn += int((~tp).sum())
        iou_all, iou_kept = iou_matrix(b, g), iou_matrix(pb, g)
        for j in range(len(g)):
            if ok[j]: status = 'détectée'
            elif len(b) and (iou_all[:, j][sc < thr] >= IOU_THR).any(): status = 'score trop bas'
            elif len(pb) and iou_kept[:, j].max() >= 0.1: status = 'mal localisée'
            else: status = 'non détectée'
            gt_rows.append(dict(model=MODEL_NAME, seed=seed, stem=s, gt_idx=j, detected=bool(ok[j]), status=status,
                                best_iou=float(iou_all[:, j].max()) if len(b) else 0.0))
        for i in np.where(~tp)[0]:
            if len(g) == 0: kind = 'image sans fracture'
            else:
                mx = iou_kept[i].max()
                kind = 'doublon' if mx >= IOU_THR else ('localisation imprécise' if mx >= 0.1 else 'autre région')
            fp_rows.append(dict(model=MODEL_NAME, seed=seed, stem=s, score=float(ps[i]), kind=kind))
    FN = n_gt - TP; P, R = TP / (TP + FPn + 1e-9), TP / (TP + FN + 1e-9)
    runs.append(dict(model=MODEL_NAME, seed=seed, seuil=thr, TP=TP, FP=FPn, FN=FN, precision=P, rappel=R, F1=2 * P * R / (P + R + 1e-9)))

gt_res = pd.DataFrame(gt_rows).merge(gt_df, on=['stem', 'gt_idx']); fp_res = pd.DataFrame(fp_rows); runs = pd.DataFrame(runs)
tag = 'fasterrcnn'
gt_res.to_csv(f'{ERR_DIR}/detail_fractures_{tag}.csv', index=False); fp_res.to_csv(f'{ERR_DIR}/detail_faux_positifs_{tag}.csv', index=False)
runs.to_csv(f'{ERR_DIR}/bilan_par_graine_{tag}.csv', index=False); gt_df.to_csv(f'{ERR_DIR}/fractures_annotees.csv', index=False)

# ── Tableaux de synthèse (écart-type avec correction de Bessel, ddof=1, comme dans l'article) ──
ms = lambda s: f'{s.mean():.3f} ± {s.std(ddof=1):.3f}'
nS = len(SEEDS)
out = [('Tableau A. Bilan par graine', runs.round(3).set_index('seed')),
       ('Tableau A bis. Bilan moyen (moyenne ± écart-type)', pd.DataFrame({
           'seuil': [runs.seuil.mean()], 'TP': [runs.TP.mean()], 'FP': [runs.FP.mean()], 'FN': [runs.FN.mean()],
           'précision': [ms(runs.precision)], 'rappel': [ms(runs.rappel)], 'F1': [ms(runs.F1)]}, index=[MODEL_NAME]).round(2)),
       ('Tableau B. Statut des fractures annotées (nombre moyen par graine)',
        (gt_res.groupby('status').size() / nS).round(1).to_frame(MODEL_NAME))]
for fac in ['région', 'taille', 'matériel', 'fractures multiples']:
    r = ((1 - gt_res.groupby([fac, 'seed']).detected.mean().groupby(fac).mean()) * 100).round(1).to_frame('FN (%)')
    r.insert(0, 'n fractures', gt_df.groupby(fac).size()); out.append((f'Tableau C. Taux de faux négatifs selon : {fac}', r))
if len(fp_res):
    out.append(('Tableau D. Nature des faux positifs (nombre moyen par graine)', (fp_res.groupby('kind').size() / nS).round(1).to_frame(MODEL_NAME)))
never = gt_res.groupby(['stem', 'gt_idx']).detected.sum(); never = int((never == 0).sum())
with open(f'{ERR_DIR}/synthese_{tag}.md', 'w', encoding='utf-8') as f:
    for titre, t in out:
        f.write(f'### {titre}\n\n{t.to_markdown() if hasattr(t, "to_markdown") else t.to_string()}\n\n'); print(f'\n{titre}\n{t}')
    f.write(f'Fractures jamais détectées (aucune graine) : {never}\n')
print(f'\nFractures jamais détectées par aucune des {nS} graines : {never} / {len(gt_df)}')
print('Contrôle TP + FN = fractures annotées :', bool(((runs.TP + runs.FN) == n_gt).all()), f'({n_gt})')

# ── Images d'exemples (vert = annotation, rouge = détection) ──
ex = f'{ERR_DIR}/exemples_{tag}'; os.makedirs(ex, exist_ok=True)
s0 = SEEDS[0]; thr0 = cache['thr'][s0]
nv = gt_res.groupby(['stem', 'gt_idx']).detected.sum(); nv = list(nv[nv == 0].index.get_level_values(0).unique()[:5])
tf = list(fp_res[fp_res.seed == s0].sort_values('score', ascending=False).drop_duplicates('stem').stem[:3]) if len(fp_res) else []
for s in nv + tf:
    img = cv2.imread(TEST_P[s])
    for b in TEST_GT[s]: cv2.rectangle(img, tuple(map(int, b[:2])), tuple(map(int, b[2:])), (0, 200, 0), 2)
    for bb, c in zip(*cache['test'][s0][s]):
        if c >= thr0:
            cv2.rectangle(img, tuple(map(int, bb[:2])), tuple(map(int, bb[2:])), (0, 0, 255), 2)
            cv2.putText(img, f'{c:.2f}', (int(bb[0]), max(int(bb[1]) - 4, 10)), 0, 0.5, (0, 0, 255), 1)
    cv2.imwrite(f'{ex}/{s}.png', img)
print(f'\n✅ Terminé. Résultats dans {ERR_DIR} → envoie-moi synthese_{tag}.md, bilan_par_graine_{tag}.csv et detail_fractures_{tag}.csv')


## 📐 Cellule 9 — Précision, rappel et F1 (boîtes) — Faster R-CNN, 5 graines

À lancer après les Cellules 1 à 6 (réutilise `SAVE_DIR`, `SEEDS`, `build_model`, `FracAtlasDetectionDataset`, `collate_fn`, `device`). Poids : `SAVE_DIR/seed_X/best.pth`.

- **Vérification préalable** : les 5 entraînements doivent être terminés (`training_done.json`).
- Les sorties du modèle sont réglées comme dans la Cellule 6 et en section 2.5 : `score_thresh = 0,001`, `detections_per_img = 300`, NMS de torchvision à 0,5.
- mAP@50 et AUC sont recalculés et comparés à ceux de la Cellule 6 ; le mAP@50 est aussi donné avec 300 boîtes par image (torchmetrics n'en compte que 100 par défaut).
- Une prédiction est un vrai positif si son IoU avec une boîte annotée est ≥ 0,5 (appariement glouton par confiance décroissante).
- Le seuil de confiance maximise le **F1 de validation**, sur la grille 0,05–0,95 par pas de 0,05 (identique à la section 2.5 et à l'analyse des erreurs) : TP, FP, FN et F1 doivent retrouver le tableau A bis de l'analyse des erreurs.
- Images normales : FP par image normale et spécificité image à ce seuil. Moyenne ± écart-type avec **ddof = 1**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 9 — Précision / Rappel / F1 au niveau des boîtes (Faster R-CNN, 5 graines)
# Seuil de confiance choisi sur la VALIDATION (maximise le F1), appliqué au TEST.
# Poids utilisés : SAVE_DIR/seed_X/best.pth (entraînement de la Cellule 5)
# À lancer après les Cellules 1 à 6 (réutilise build_model, FracAtlasDetectionDataset, collate_fn, device).
# Réglages de sortie identiques à la Cellule 6 : score_thresh = 0,001, detections_per_img = 300, NMS torchvision (0,5).
# ═══════════════════════════════════════════════════════════════════════════
import os, json, gc
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from torchmetrics.detection.mean_ap import MeanAveragePrecision

MODEL_DIR = SAVE_DIR            # dossier FasterRCNN (seed_X/best.pth)
RUN_SEEDS = list(SEEDS)         # mêmes 5 graines que la Cellule 1
IOU_MATCH = 0.5                 # TP si IoU >= 0.5 avec une boîte annotée
CONF_MIN  = 0.001               # section 2.5 : on conserve les prédictions de score >= 0.001...
MAX_DET   = 300                 # ... dans la limite de 300 boîtes par image
CHUNK     = 4                   # images par lot à l'inférence (Faster R-CNN agrandit à 800 px en interne)
THR_GRID  = np.round(np.arange(0.05, 0.951, 0.05), 2)   # même grille que la section 2.5
# égalité de F1 → np.argmax retient le seuil le plus faible, comme décrit en 2.5


def _iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def _collect(model, split):
    """Prédictions (score >= CONF_MIN, 300 max) + vérité terrain, image par image."""
    ds = FracAtlasDetectionDataset(split, train=False)
    loader = DataLoader(ds, batch_size=CHUNK, shuffle=False, collate_fn=collate_fn, num_workers=2)
    data, preds, targets_all = [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, targets in loader:
            outs = model([im.to(device) for im in imgs])
            for o, t in zip(outs, targets):
                boxes = o['boxes'].cpu().numpy().reshape(-1, 4)
                scores = o['scores'].cpu().numpy()
                labels = o['labels'].cpu().numpy()
                order = np.argsort(-scores)
                boxes, scores, labels = boxes[order], scores[order], labels[order]
                gt = t['boxes'].numpy().reshape(-1, 4)
                data.append({'scores': scores, 'n_gt': len(gt), 'iou': _iou_matrix(boxes, gt)})
                preds.append({'boxes': torch.as_tensor(boxes, dtype=torch.float32),
                              'scores': torch.as_tensor(scores, dtype=torch.float32),
                              'labels': torch.as_tensor(labels, dtype=torch.int64)})
                targets_all.append({'boxes': t['boxes'], 'labels': t['labels']})
    return data, preds, targets_all


def _counts(data, thr):
    """TP/FP/FN au niveau des boîtes, appariement glouton par confiance décroissante."""
    tp = fp = fn = 0
    n_normal = fp_normal = normal_flagged = 0
    for d in data:
        n_pred = int((d['scores'] >= thr).sum())     # scores triés : les n_pred premiers
        n_gt = d['n_gt']
        taken = np.zeros(n_gt, dtype=bool)
        tp_i = 0
        for i in range(n_pred):
            if n_gt == 0:
                break
            ious = np.where(taken, -1.0, d['iou'][i])
            j = int(np.argmax(ious))
            if ious[j] >= IOU_MATCH:
                taken[j] = True
                tp_i += 1
        tp += tp_i; fp += n_pred - tp_i; fn += n_gt - tp_i
        if n_gt == 0:
            n_normal += 1; fp_normal += n_pred; normal_flagged += int(n_pred > 0)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'TP': tp, 'FP': fp, 'FN': fn, 'Precision': prec, 'Rappel': rec, 'F1': f1,
            'FP_par_image_normale': fp_normal / n_normal if n_normal else float('nan'),
            'Specificite_image': 1 - normal_flagged / n_normal if n_normal else float('nan')}


def _image_auc(data, preds_view):
    y_true = np.array([int(d['n_gt'] > 0) for d in data])
    y_score = np.array([float(p['scores'].max()) if len(p['scores']) else 0.0 for p in preds_view])
    return float(roc_auc_score(y_true, y_score))


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Vérification avant de lancer : entraînements terminés, poids présents ──
print(f'Optimiseur={OPTIMIZER} | lr0={LR0} | batch={BATCH_SIZE} (sans accumulation : batch effectif = {BATCH_SIZE}) | patience={PATIENCE}')
print(f"{'seed':<6}{'epochs faites':<15}{'meilleure val_loss'}")
for seed in RUN_SEEDS:
    ckpt = os.path.join(MODEL_DIR, f'seed_{seed}', 'best.pth')
    done = os.path.join(MODEL_DIR, f'seed_{seed}', 'training_done.json')
    assert os.path.exists(ckpt), f'Poids introuvables (seed {seed}) : {ckpt}'
    assert os.path.exists(done), f"training_done.json absent (seed {seed}) : entraînement non terminé ou interrompu."
    d = json.load(open(done))
    print(f"{seed:<6}{str(d.get('epochs_done')):<15}{d.get('best_val_loss')}")
print()

detect_by_seed = {}
for seed in RUN_SEEDS:
    out_path = os.path.join(MODEL_DIR, f'metriques_detection_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            detect_by_seed[seed] = json.load(f)
        print(f'✅ Seed {seed} : métriques de détection déjà calculées, chargées.')
        continue

    print(f'\n📐 Seed {seed} — précision / rappel / F1...')
    model = build_model()
    model.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'seed_{seed}', 'best.pth'), map_location=device))
    model.to(device)
    # réglages de sortie de la section 2.5 (le NMS reste celui de torchvision : nms_thresh = 0.5)
    model.roi_heads.score_thresh = CONF_MIN
    model.roi_heads.detections_per_img = MAX_DET
    print(f'   réglages de sortie : score_thresh={model.roi_heads.score_thresh}, '
          f'detections_per_img={model.roi_heads.detections_per_img}, nms_thresh={model.roi_heads.nms_thresh}')

    # 1) seuil de confiance qui maximise le F1 (boîtes) sur la validation
    val_data, _, _ = _collect(model, 'val')
    f1_val = np.array([_counts(val_data, t)['F1'] for t in THR_GRID])
    thr = float(THR_GRID[int(np.argmax(f1_val))])

    # 2) métriques sur le test à ce seuil
    test_data, test_preds, test_targets = _collect(model, 'test')
    m = _counts(test_data, thr)

    # 3) mAP et AUC : (a) mAP avec 300 boîtes par image (section 2.5), (b) exactement comme la Cellule 6
    #    (torchmetrics compte par défaut les 100 meilleures boîtes par image)
    m_full = MeanAveragePrecision(max_detection_thresholds=[1, 10, MAX_DET])
    m_full.update(test_preds, test_targets)
    r = m_full.compute()
    map50, map5095 = float(r['map_50']), float(r['map'])
    m_c6 = MeanAveragePrecision(iou_thresholds=[0.5])
    m_c6.update(test_preds, test_targets)
    map50_c6 = float(m_c6.compute()['map'])
    auc = _image_auc(test_data, test_preds)

    # Contrôle de cohérence avec ce que la Cellule 6 avait enregistré
    prev = os.path.join(MODEL_DIR, f'metriques_seed{seed}.json')
    if os.path.exists(prev):
        _p = json.load(open(prev))['Fracture']
        if abs(_p['mAP50'] - map50_c6) > 0.002 or abs(_p['AUC'] - auc) > 0.002:
            print(f"   ⚠️ valeurs enregistrées (mAP50={_p['mAP50']:.4f}, AUC={_p['AUC']:.4f}) ≠ recalcul "
                  f"(mAP50={map50_c6:.4f}, AUC={auc:.4f}) : vérifier que ce sont les mêmes poids et les mêmes réglages de sortie.")
        else:
            print('   ✅ mAP50 et AUC retrouvés (identiques à la Cellule 6)')
    if map50 < 0.1:
        print("   ⚠️ mAP50 anormalement bas : checkpoint probablement issu d'un entraînement interrompu.")

    res = {'seuil_F1_val': thr, 'F1_val_max': round(float(f1_val.max()), 4),
           'mAP50': round(map50, 4), 'mAP50_95': round(map5095, 4), 'AUC': round(auc, 4),
           'mAP50_100boites': round(map50_c6, 4)}
    res.update({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()})
    with open(out_path, 'w') as f:
        json.dump(res, f, indent=2)
    detect_by_seed[seed] = res
    print(f"   seuil={thr:.2f} | P={res['Precision']:.3f} R={res['Rappel']:.3f} F1={res['F1']:.3f} "
          f"| FP/img normale={res['FP_par_image_normale']:.3f} | mAP50-95={map5095:.3f}")
    print(f"   mAP50 : {map50:.4f} (300 boîtes/image) vs {map50_c6:.4f} (100 boîtes, comme la Cellule 6) | AUC : {auc:.4f}")

    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Agrégation : moyenne ± écart-type (ddof=1, correction de Bessel, n = nb de graines) ──
keys = ['mAP50', 'mAP50_100boites', 'AUC', 'Precision', 'Rappel', 'F1',
        'mAP50_95', 'TP', 'FP', 'FN', 'FP_par_image_normale', 'Specificite_image', 'seuil_F1_val']
agg_det = {'n_seeds': len(RUN_SEEDS), 'ddof': 1, 'poids': 'FasterRCNN (batch=%d)' % BATCH_SIZE}
print('\nFaster R-CNN — détection (boîtes, IoU >= 0.5), moyenne ± é.-t. (ddof=1) sur', len(RUN_SEEDS), 'graines')
for k in keys:
    v = np.array([detect_by_seed[s][k] for s in RUN_SEEDS], dtype=float)
    agg_det[k] = {'mean': round(float(v.mean()), 4), 'std': round(float(v.std(ddof=1)), 4)}
    print(f"  {k:<26s} {v.mean():.3f} ± {v.std(ddof=1):.3f}")
with open(os.path.join(MODEL_DIR, 'resultats_detection_agreges.json'), 'w') as f:
    json.dump(agg_det, f, indent=2)
